# Hard Number Theory Control Probe

This is a negative-control check, not part of the official benchmark. The questions are proof-style and non-binary, but they are harder elementary number-theory questions rather than stochastic-process questions.


In [ ]:
!pip install -q -U mlx-lm pandas matplotlib tqdm


In [ ]:
from pathlib import Path
import importlib
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from mlx_lm import generate, load
from tqdm.auto import tqdm


In [ ]:
PROJECT_ROOT = Path("/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT


In [ ]:
import training_eval.eval_utils as eval_utils

importlib.reload(eval_utils)

GRADING_POLICY = eval_utils.GRADING_POLICY
extract_answer = eval_utils.extract_answer
is_correct = eval_utils.is_correct
load_jsonl_records = eval_utils.load_jsonl_records
rows_to_frame = eval_utils.rows_to_frame
save_results = eval_utils.save_results
summarize_accuracy = eval_utils.summarize_accuracy


In [ ]:
MODEL_NAME = "Qwen/Qwen3-1.7B-MLX-bf16"
DATA_DIR = PROJECT_ROOT / "benchmark" / "data" / "ood_math_control"
RESULT_ROOT = PROJECT_ROOT / "results" / "ood_math_control_number_theory_hard"
ADAPTER_ROOT = PROJECT_ROOT / "results" / "fine_tunes" / "qwen3_1_7b_lora"
MAX_NEW_TOKENS = 768


In [ ]:
RUNS = {
    "Qwen3 1.7B base": None,
    "LoRA algorithmic_scaffold_v2": ADAPTER_ROOT / "algorithmic_scaffold_v2" / "adapters",
    "LoRA algorithmic_scaffold_v3_5": ADAPTER_ROOT / "algorithmic_scaffold_v3_5" / "adapters",
    "LoRA algorithmic_scaffold_v3_5_lora12": ADAPTER_ROOT / "algorithmic_scaffold_v3_5_lora12" / "adapters",
}


In [ ]:
records = load_jsonl_records(DATA_DIR, pattern="*.jsonl")
len(records)


In [ ]:
pd.DataFrame(records).groupby(["answer_type", "problem_type"]).size().rename("count").reset_index()


In [ ]:
FINE_TUNE_SYSTEM_MESSAGE = """You solve discrete stochastic-process problems. Give concise reasoning, then end with exactly one final answer block: Final answer:
<answer>
{...}
</answer>. Do not write anything after </answer>."""


In [ ]:
def make_fine_tuned_chat_messages(problem):
    return [
        {"role": "system", "content": FINE_TUNE_SYSTEM_MESSAGE},
        {"role": "user", "content": problem},
    ]


In [ ]:
def make_qwen_prompt(tokenizer, problem):
    messages = make_fine_tuned_chat_messages(problem)
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


In [ ]:
def load_run_model(adapter_path):
    if adapter_path is None:
        return load(MODEL_NAME)
    return load(MODEL_NAME, adapter_path=str(adapter_path))


In [ ]:
def generate_answer(model, tokenizer, problem):
    prompt = make_qwen_prompt(tokenizer, problem)
    return generate(model, tokenizer, prompt=prompt, max_tokens=MAX_NEW_TOKENS, verbose=False)


In [ ]:
def evaluate_run(run_label, adapter_path, records):
    model, tokenizer = load_run_model(adapter_path)
    rows = []

    for record in tqdm(records, desc=run_label):
        raw_output = generate_answer(model, tokenizer, record["problem"])
        predicted = extract_answer(raw_output, record["canonical_answer"])
        rows.append({
            "run": run_label,
            "id": record["id"],
            "family": record["family"],
            "problem_type": record["problem_type"],
            "difficulty": record["difficulty"],
            "answer_type": record["answer_type"],
            "problem": record["problem"],
            "canonical_answer": record["canonical_answer"],
            "raw_output": raw_output,
            "predicted_answer": predicted,
            "correct": is_correct(predicted, record["canonical_answer"]),
        })

    return rows


In [ ]:
def save_run(run_label, rows):
    safe_label = run_label.lower().replace(" ", "_").replace("/", "_")
    result_dir = RESULT_ROOT / safe_label
    df = rows_to_frame(rows)
    metrics = summarize_accuracy(df)
    metrics.update({
        "model": MODEL_NAME,
        "run": run_label,
        "dataset": "benchmark/data/ood_math_control/*.jsonl",
        "grading_policy": GRADING_POLICY,
    })
    save_results(rows, result_dir, metrics)
    return result_dir


## Run Probe

This evaluates the base model and any listed adapters whose folders exist. Re-running overwrites the same OOD result folders.


In [ ]:
all_rows = []

for run_label, adapter_path in RUNS.items():
    if adapter_path is not None and not adapter_path.exists():
        print(f"Skipping missing adapter: {run_label} -> {adapter_path}")
        continue
    rows = evaluate_run(run_label, adapter_path, records)
    save_run(run_label, rows)
    all_rows.extend(rows)

df = rows_to_frame(all_rows)
df.head()


## Tables


In [ ]:
summary_df = (
    df.groupby("run")["correct"]
    .agg(accuracy="mean", correct="sum", count="count")
    .reset_index()
    .sort_values("accuracy", ascending=False)
)

answer_type_df = (
    df.groupby(["run", "answer_type"])["correct"]
    .agg(accuracy="mean", correct="sum", count="count")
    .reset_index()
    .sort_values(["answer_type", "accuracy"], ascending=[True, False])
)

problem_type_df = (
    df.groupby(["run", "answer_type", "problem_type"])["correct"]
    .agg(accuracy="mean", correct="sum", count="count")
    .reset_index()
    .sort_values(["answer_type", "problem_type", "accuracy"], ascending=[True, True, False])
)

display(summary_df)
display(answer_type_df)
problem_type_df


## Charts


In [ ]:
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ORDER = summary_df["run"].tolist()


In [ ]:
def plot_answer_type_accuracy(answer_type_df):
    plot_df = answer_type_df.pivot(index="run", columns="answer_type", values="accuracy")
    plot_df = plot_df.reindex([run for run in RUN_ORDER if run in plot_df.index])
    ax = plot_df.plot(kind="bar", figsize=(10, 4.5), ylim=(0, 1), width=0.82)
    ax.set_title("Hard number-theory control accuracy by answer type")
    ax.set_xlabel("")
    ax.set_ylabel("Accuracy")
    ax.axhline(0.5, color="0.75", linestyle="--", linewidth=1)
    ax.tick_params(axis="x", rotation=25)
    plt.tight_layout()
    plt.savefig(RESULT_ROOT / "hard_number_theory_control_accuracy_by_answer_type.png", dpi=180)
    plt.show()


In [ ]:
plot_answer_type_accuracy(answer_type_df)


In [ ]:
def plot_problem_type_heatmap(problem_type_df, answer_type):
    plot_df = problem_type_df[problem_type_df["answer_type"] == answer_type]
    if plot_df.empty:
        print(f"No {answer_type} rows to plot.")
        return

    matrix = plot_df.pivot(index="problem_type", columns="run", values="accuracy")
    matrix = matrix[[run for run in RUN_ORDER if run in matrix.columns]]

    fig, ax = plt.subplots(figsize=(max(8, 0.8 * len(matrix.columns)), max(4, 0.35 * len(matrix.index))))
    image = ax.imshow(matrix.fillna(0), vmin=0, vmax=1, aspect="auto", cmap="RdYlGn")
    ax.set_title(f"Hard number-theory control {answer_type.replace('_', '-')} accuracy by problem type")
    ax.set_xticks(range(len(matrix.columns)))
    ax.set_xticklabels(matrix.columns, rotation=25, ha="right")
    ax.set_yticks(range(len(matrix.index)))
    ax.set_yticklabels(matrix.index)

    for i, problem_type in enumerate(matrix.index):
        for j, run in enumerate(matrix.columns):
            value = matrix.loc[problem_type, run]
            if pd.notna(value):
                ax.text(j, i, f"{value:.0%}", ha="center", va="center", fontsize=8)

    fig.colorbar(image, ax=ax, label="Accuracy")
    plt.tight_layout()
    plt.savefig(RESULT_ROOT / f"hard_number_theory_control_accuracy_by_problem_type_{answer_type}.png", dpi=180)
    plt.show()


In [ ]:
for answer_type in sorted(problem_type_df["answer_type"].unique()):
    plot_problem_type_heatmap(problem_type_df, answer_type)


In [ ]:
summary_df.to_csv(RESULT_ROOT / "summary.csv", index=False)
answer_type_df.to_csv(RESULT_ROOT / "by_answer_type.csv", index=False)
problem_type_df.to_csv(RESULT_ROOT / "by_problem_type.csv", index=False)
df.to_csv(RESULT_ROOT / "all_outputs.csv", index=False)
RESULT_ROOT


## Inspect Misses


In [ ]:
df.loc[
    ~df["correct"],
    ["run", "id", "answer_type", "problem_type", "canonical_answer", "predicted_answer", "raw_output"],
]
